# Day 5: Real fMRI Data — Download, Explore, and Train
**Goal:** Get actual brain recordings and train the memory module against real neural targets.

The Algonauts 2025 dataset stores fMRI as `.h5` files with shape `(N_samples, 1000_parcels)`.
We'll download data for one subject, explore the structure, and run the first real training.

**Runtime:** A100 GPU

In [ ]:
!pip install numpy==2.2.6 -q
# RESTART RUNTIME (Runtime > Restart runtime) then continue from next cell

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
CACHE_DIR = os.path.join(PROJECT_DIR, 'cache')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
FMRI_DIR = os.path.join(DATA_DIR, 'algonauts_fmri')
for d in [CACHE_DIR, RESULTS_DIR, DATA_DIR, CHECKPOINTS_DIR, FMRI_DIR]:
    os.makedirs(d, exist_ok=True)

os.system('pip install git+https://github.com/facebookresearch/tribev2.git -q')
os.system('pip install nilearn h5py -q')
os.system('git clone https://github.com/Mrabbi3/memory-augmented-brain-encoding.git /content/my_repo 2>/dev/null')

import sys
sys.path.insert(0, '/content/my_repo/src')

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
except Exception:
    from huggingface_hub import login
    login()

import torch, numpy as np
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete!')

---
## 1. Install DataLad and Download fMRI Data

The Algonauts 2025 data requires DataLad (a git-based data version control system).
We'll install it and download just ONE subject's data to keep it manageable.

In [ ]:
# Install DataLad and git-annex
!apt-get install -y git-annex 2>/dev/null | tail -1
!pip install datalad -q

# Verify installation
!datalad --version
!git-annex version | head -1
print('DataLad installed!')

In [ ]:
# Check if data already downloaded (saved in Google Drive from previous sessions)
fmri_flag = os.path.join(FMRI_DIR, 'sub01_downloaded.flag')

if os.path.exists(fmri_flag):
    print('fMRI data already downloaded! Skipping...')
else:
    print('Downloading Algonauts 2025 fMRI data for Subject 1...')
    print('This may take 10-20 minutes.')
    print()
    
    # Clone the dataset structure (fast - just symbolic links)
    os.chdir('/content')
    !datalad install -r -s https://github.com/courtois-neuromod/algonauts_2025.competitors.git algonauts_data 2>&1 | tail -5
    
    # Download only subject 1 fMRI data
    os.chdir('/content/algonauts_data')
    !datalad get fmri/sub-01/ 2>&1 | tail -10
    
    # Copy h5 files to Google Drive for persistence
    import glob, shutil
    h5_files = glob.glob('/content/algonauts_data/fmri/sub-01/**/*.h5', recursive=True)
    for f in h5_files:
        dest = os.path.join(FMRI_DIR, os.path.basename(f))
        if not os.path.exists(dest):
            shutil.copy2(f, dest)
            print(f'Copied: {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)')
    
    with open(fmri_flag, 'w') as f:
        f.write('downloaded')
    print('\nfMRI data saved to Google Drive!')

---
## 2. Explore the fMRI Data Structure

In [ ]:
import h5py
import glob

# Find all h5 files
h5_files = sorted(glob.glob(os.path.join(FMRI_DIR, '*.h5')))
print(f'Found {len(h5_files)} h5 files in Google Drive:')
for f in h5_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

# If no h5 files found in Drive, try the datalad directory
if len(h5_files) == 0:
    h5_files = sorted(glob.glob('/content/algonauts_data/fmri/sub-01/**/*.h5', recursive=True))
    print(f'\nFound {len(h5_files)} h5 files in datalad directory:')
    for f in h5_files:
        if os.path.exists(f) and not os.path.islink(f):
            size_mb = os.path.getsize(f) / 1e6
            print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

In [ ]:
# Open the first h5 file and examine its structure
if len(h5_files) > 0:
    with h5py.File(h5_files[0], 'r') as f:
        print(f'File: {os.path.basename(h5_files[0])}')
        print(f'Number of datasets (episodes): {len(f.keys())}')
        print(f'\nFirst 10 datasets:')
        for i, key in enumerate(sorted(f.keys())[:10]):
            data = f[key][:]
            print(f'  {key}: shape={data.shape}, dtype={data.dtype}')
            print(f'    → {data.shape[0]} timepoints × {data.shape[1]} brain parcels')
            print(f'    → {data.shape[0] * 1.49:.0f} seconds of brain recording')
else:
    print('No h5 files found. DataLad download may have failed.')
    print('Try running the download cell again, or check the error messages above.')
    print()
    print('ALTERNATIVE: We can generate synthetic fMRI data to test the pipeline.')
    print('Run the next cell to create synthetic data.')

In [ ]:
# === FALLBACK: Generate synthetic fMRI data if download failed ===
# This lets you test the full training pipeline regardless of data access
# Shape matches real Algonauts format: (N_samples, 1000_parcels)

SYNTHETIC = len(h5_files) == 0

if SYNTHETIC:
    print('Generating synthetic fMRI data for pipeline testing...')
    
    synthetic_path = os.path.join(FMRI_DIR, 'synthetic_fmri.h5')
    
    with h5py.File(synthetic_path, 'w') as f:
        # Create 10 "episodes" of varying length
        for ep in range(10):
            n_samples = np.random.randint(200, 500)  # 5-12 minutes
            n_parcels = 1000
            
            # Generate semi-realistic fMRI-like signals
            # Low-frequency fluctuations + noise (mimics real BOLD signal)
            t = np.linspace(0, n_samples * 1.49, n_samples)
            base_signal = np.zeros((n_samples, n_parcels))
            for p in range(n_parcels):
                freq = np.random.uniform(0.01, 0.1)  # Low freq oscillations
                phase = np.random.uniform(0, 2 * np.pi)
                base_signal[:, p] = np.sin(2 * np.pi * freq * t + phase)
            
            noise = np.random.randn(n_samples, n_parcels) * 0.5
            fmri_data = (base_signal + noise).astype(np.float32)
            
            f.create_dataset(f'episode_{ep:02d}', data=fmri_data)
    
    h5_files = [synthetic_path]
    print(f'Created synthetic data: {synthetic_path}')
    print('NOTE: Results from synthetic data are for pipeline validation only.')
else:
    print('Using REAL fMRI data!')

---
## 3. Load fMRI Data Into Memory

In [ ]:
# Load all episodes from the h5 file(s)
all_fmri = []
episode_info = []

for h5_path in h5_files:
    with h5py.File(h5_path, 'r') as f:
        for key in sorted(f.keys()):
            data = f[key][:].astype(np.float32)
            all_fmri.append(data)
            episode_info.append({
                'file': os.path.basename(h5_path),
                'key': key,
                'n_samples': data.shape[0],
                'n_parcels': data.shape[1],
                'duration_sec': data.shape[0] * 1.49,
            })

print(f'Loaded {len(all_fmri)} episodes')
total_samples = sum(d.shape[0] for d in all_fmri)
total_minutes = sum(info['duration_sec'] for info in episode_info) / 60
print(f'Total: {total_samples:,} timepoints ({total_minutes:.1f} minutes of brain recording)')
print(f'Brain parcels: {all_fmri[0].shape[1]}')

# Show first 5 episodes
for info in episode_info[:5]:
    print(f"  {info['key']}: {info['n_samples']} samples ({info['duration_sec']:.0f}s)")

---
## 4. Load TRIBE v2 Model + Memory Module

In [ ]:
from tribev2 import TribeModel
from memory import MemoryAttention, MemoryBuffer, MemoryAugmentedEncoder

print('Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
brain_model = model._model
device = brain_model.device

# Freeze TRIBE v2
for param in brain_model.parameters():
    param.requires_grad = False

# Create memory encoder
mem_encoder = MemoryAugmentedEncoder(
    brain_model=brain_model,
    buffer_size=100,
    top_k=5,
    num_heads=8,
)

# Load Day 4 checkpoint if available
ckpt_path = os.path.join(CHECKPOINTS_DIR, 'memory_module_final.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    mem_encoder.memory_attention.load_state_dict(ckpt['memory_attention_state_dict'])
    print('Loaded Day 4 checkpoint!')
else:
    print('No checkpoint found, starting fresh.')

trainable = sum(p.numel() for p in mem_encoder.memory_attention.parameters())
print(f'Trainable params: {trainable:,}')
print(f'Model on: {device}')

---
## 5. Training With Real fMRI Targets

**Key change from Day 4:** Instead of matching vanilla predictions,
we now train against ACTUAL brain recordings.

The pipeline:
1. For each episode, generate a test video (since we need stimulus→prediction)
2. Run TRIBE v2 + memory to get predictions at 1000 parcels
3. Compare to real fMRI at 1000 parcels
4. Backprop through memory attention only

**Important:** The real Algonauts data has 1000 parcels, but TRIBE v2 predicts
20,484 vertices. We need to aggregate vertices to parcels for comparison.
For now, we'll use the first 1000 dimensions as a proxy (the actual mapping
requires the Schaefer atlas, which we'll add in a later session).

**Alternative simpler approach:** We train the memory module's latent
representations directly, using fMRI parcels as auxiliary targets via a
lightweight projection layer.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Create a lightweight projection from TRIBE v2's latent space to fMRI parcels
# This bypasses the vertex→parcel mapping problem
n_parcels = all_fmri[0].shape[1]  # 1000
hidden_dim = brain_model.config.hidden  # 1152

parcel_projector = nn.Linear(hidden_dim, n_parcels).to(device)
print(f'Parcel projector: {hidden_dim} → {n_parcels}')
proj_params = sum(p.numel() for p in parcel_projector.parameters())
print(f'Projector params: {proj_params:,}')

# Total trainable: memory attention + parcel projector
all_trainable = list(mem_encoder.memory_attention.parameters()) + list(parcel_projector.parameters())
total_trainable = sum(p.numel() for p in all_trainable)
print(f'Total trainable: {total_trainable:,}')

In [ ]:
# Prepare fMRI data as torch tensors
# We'll use sliding windows of ~100 seconds (matching TRIBE v2's window size)

WINDOW_SIZE = 67  # ~100 seconds at 1.49s per sample
STRIDE = 33  # ~50% overlap

# Create windowed training samples from fMRI data
fmri_windows = []
for ep_idx, fmri_data in enumerate(all_fmri):
    n_samples = fmri_data.shape[0]
    for start in range(0, n_samples - WINDOW_SIZE, STRIDE):
        window = fmri_data[start:start + WINDOW_SIZE]  # (WINDOW_SIZE, 1000)
        fmri_windows.append({
            'data': torch.tensor(window, dtype=torch.float32),
            'episode': ep_idx,
            'start_sample': start,
        })

print(f'Created {len(fmri_windows)} training windows')
print(f'Window size: {WINDOW_SIZE} samples ({WINDOW_SIZE * 1.49:.0f}s)')
print(f'Stride: {STRIDE} samples ({STRIDE * 1.49:.0f}s)')
print(f'Each window shape: {fmri_windows[0]["data"].shape}')

In [ ]:
# === TRAINING LOOP WITH REAL fMRI ===

NUM_EPOCHS = 10
LEARNING_RATE = 3e-4
BATCH_SIZE = 4  # windows per batch

optimizer = AdamW(all_trainable, lr=LEARNING_RATE, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

mem_encoder.memory_attention.train()
parcel_projector.train()

train_losses = []
gate_values = []
best_loss = float('inf')

print('='*60)
print('TRAINING WITH REAL fMRI DATA')
print('='*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    
    # Reset memory at start of each epoch
    mem_encoder.reset_memory()
    current_episode = -1
    
    # Process windows in order (sequential for memory)
    for win_idx in range(0, len(fmri_windows), BATCH_SIZE):
        batch_windows = fmri_windows[win_idx:win_idx + BATCH_SIZE]
        
        # Reset memory when episode changes
        for w in batch_windows:
            if w['episode'] != current_episode:
                mem_encoder.reset_memory()
                current_episode = w['episode']
                break
        
        # Stack fMRI targets
        fmri_target = torch.stack([w['data'] for w in batch_windows]).to(device)
        # fmri_target shape: (B, WINDOW_SIZE, 1000)
        
        optimizer.zero_grad()
        
        # Generate synthetic transformer latents
        # (In a full pipeline, these would come from actual video features)
        # For now we use random latents as a placeholder
        B = fmri_target.shape[0]
        T = fmri_target.shape[1]
        
        with torch.no_grad():
            # Simulate transformer output
            x = torch.randn(B, T, hidden_dim).to(device)
        
        # Memory operations (TRAINABLE)
        query = x.mean(dim=1)
        retrieved = mem_encoder.memory_buffer.retrieve(query)
        x_memory = mem_encoder.memory_attention(x, retrieved)
        
        # Store in buffer
        mem_encoder.memory_buffer.store(x_memory)
        
        # Project to parcel space (TRAINABLE)
        pred_parcels = parcel_projector(x_memory)  # (B, T, 1000)
        
        # Loss: MSE between predicted and real fMRI parcels
        loss = F.mse_loss(pred_parcels, fmri_target)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / max(n_batches, 1)
    gate_val = torch.tanh(mem_encoder.memory_attention.gate).item()
    lr = optimizer.param_groups[0]['lr']
    
    train_losses.append(avg_loss)
    gate_values.append(gate_val)
    scheduler.step()
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'memory_attention': mem_encoder.memory_attention.state_dict(),
            'parcel_projector': parcel_projector.state_dict(),
            'epoch': epoch,
            'loss': avg_loss,
            'gate': gate_val,
        }, os.path.join(CHECKPOINTS_DIR, 'best_fmri_trained.pt'))
    
    elapsed = time.time() - start_time
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Loss: {avg_loss:.4f} | '
          f'Gate: {gate_val:.4f} | '
          f'LR: {lr:.2e} | '
          f'Batches: {n_batches} | '
          f'Time: {elapsed:.0f}s')

print(f'\nTraining complete! Best loss: {best_loss:.4f}')
print(f'Final gate: {gate_values[-1]:.4f}')

---
## 6. Visualize Training Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss (fMRI target)')
axes[0].grid(True, alpha=0.3)

# Gate value
axes[1].plot(range(1, NUM_EPOCHS+1), gate_values, 'r-o', markersize=4)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Gate Value')
axes[1].set_title('Memory Gate Evolution')
axes[1].set_ylim(-1, 1)
axes[1].grid(True, alpha=0.3)

# Per-parcel prediction quality (last epoch)
# Run one forward pass to get per-parcel correlations
mem_encoder.memory_attention.eval()
parcel_projector.eval()
mem_encoder.reset_memory()

all_preds = []
all_targets = []

with torch.inference_mode():
    for win_idx in range(0, min(len(fmri_windows), 50)):
        w = fmri_windows[win_idx]
        fmri_target = w['data'].unsqueeze(0).to(device)
        x = torch.randn(1, fmri_target.shape[1], hidden_dim).to(device)
        
        query = x.mean(dim=1)
        retrieved = mem_encoder.memory_buffer.retrieve(query)
        x_memory = mem_encoder.memory_attention(x, retrieved)
        mem_encoder.memory_buffer.store(x_memory)
        
        pred = parcel_projector(x_memory)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(fmri_target.cpu().numpy())

preds_cat = np.concatenate(all_preds, axis=1)[0]  # (total_T, 1000)
targets_cat = np.concatenate(all_targets, axis=1)[0]  # (total_T, 1000)

# Per-parcel correlation
from scipy.stats import pearsonr
parcel_correlations = []
for p in range(n_parcels):
    if targets_cat[:, p].std() > 1e-6 and preds_cat[:, p].std() > 1e-6:
        r, _ = pearsonr(preds_cat[:, p], targets_cat[:, p])
        parcel_correlations.append(r)
    else:
        parcel_correlations.append(0.0)

parcel_correlations = np.array(parcel_correlations)

axes[2].hist(parcel_correlations, bins=50, color='green', alpha=0.7, edgecolor='black')
axes[2].axvline(x=np.mean(parcel_correlations), color='red', linestyle='--',
               label=f'Mean R={np.mean(parcel_correlations):.3f}')
axes[2].set_xlabel('Pearson R')
axes[2].set_ylabel('Number of parcels')
axes[2].set_title('Per-Parcel Encoding Scores')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Day 5: Training with Real fMRI Data', fontsize=14, y=1.02)
plt.tight_layout()

fig_path = os.path.join(RESULTS_DIR, 'day5_fmri_training.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved to {fig_path}')
plt.show()

print(f'\nMean parcel correlation: {np.mean(parcel_correlations):.4f}')
print(f'Max parcel correlation: {np.max(parcel_correlations):.4f}')
print(f'Parcels with R > 0.1: {(parcel_correlations > 0.1).sum()}')
print(f'Parcels with R > 0.2: {(parcel_correlations > 0.2).sum()}')

---
## 7. Save Session

In [ ]:
import json
from datetime import datetime

session = {
    'date': datetime.now().isoformat(),
    'notebook': '05_real_fmri_training',
    'synthetic_data': SYNTHETIC,
    'n_episodes': len(all_fmri),
    'total_samples': int(total_samples),
    'n_windows': len(fmri_windows),
    'n_parcels': int(n_parcels),
    'epochs': NUM_EPOCHS,
    'final_loss': float(train_losses[-1]),
    'best_loss': float(best_loss),
    'final_gate': float(gate_values[-1]),
    'mean_parcel_r': float(np.mean(parcel_correlations)),
    'max_parcel_r': float(np.max(parcel_correlations)),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'status': 'completed'
}

log_path = os.path.join(RESULTS_DIR, 'session_log.json')
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)
else:
    logs = []
logs.append(session)
with open(log_path, 'w') as f:
    json.dump(logs, f, indent=2)

data_type = 'SYNTHETIC' if SYNTHETIC else 'REAL'
print(f'Day 5 Summary ({data_type} fMRI data)')
print('='*50)
print(f'Episodes: {len(all_fmri)}')
print(f'Training windows: {len(fmri_windows)}')
print(f'Loss: {train_losses[0]:.4f} → {train_losses[-1]:.4f}')
print(f'Gate: {gate_values[0]:.4f} → {gate_values[-1]:.4f}')
print(f'Mean parcel R: {np.mean(parcel_correlations):.4f}')
print(f'\nAll saved to Google Drive!')

---
## What You Built Today

1. **DataLad setup on Colab** for downloading Algonauts fMRI data
2. **h5 data loading** — explored the actual brain recording format
3. **Parcel projector** — lightweight layer mapping latents to 1000 brain parcels
4. **Real fMRI training** — first training against actual brain recordings
5. **Per-parcel encoding scores** — measuring which brain regions improve

**Next (Day 6):** Run the full pipeline with TRIBE v2 features from actual videos
(instead of random latents) to get scientifically meaningful encoding scores.
Then compare memory vs no-memory per brain region.